# Geothermal Lithium Prediction — Full Reproducible Pipeline

Reproduces the main results of:

> Tut Haklidir, F.S. and Haklidir, M. (2026). *Predicting Dissolved Lithium in Geothermal Fluids from Routine Hydrogeochemistry: A Machine Learning Framework with Calibrated Uncertainty Quantification.* Earth Science Informatics.

## Sections

- **Part A** — Main GroupKFold cross-validation across DNN, Ridge, RF, XGB, and ElasticNet baselines (Section 4.2, Tables 3–5)
- **Part B** — Manuscript figures (Fig. 7, 8, 10, 11, 12, plus DNN training curves)
- **Part C** — Reviewer-requested experiments (GroupKFold diagnostics, DNN complexity report, DNN vs Ridge bootstrap, fold-1 diagnostics, Mg²⁺ and feature-engineering ablations, empirical-coverage binomial CI)
- **Part D** — Headline numbers for the paper text
- **Part E** — Save all predictions and metrics to disk

## How to run

1. Place `training_dataset_lit.csv` and `testing_dataset_lit.csv` in the same directory as this notebook (or use the `data/` folder of the repository).
2. Install dependencies: `pip install -r requirements.txt`
3. Run cells top-to-bottom.

All output files are written under a timestamped `FULL_RERUN_<timestamp>/` directory.

## Reproducibility

Random seed 42 is set across NumPy, Python `random`, and TensorFlow. Hyperparameters match Appendix A.2 of the manuscript exactly.

In [ ]:
# Install required packages (uncomment if running on a fresh environment)
# !pip install xgboost shap tensorflow scikit-learn pandas matplotlib scipy

import os, re, random, warnings, datetime, shutil
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV, ElasticNet
from xgboost import XGBRegressor
import shap

import tensorflow as tf
from tensorflow.keras import Sequential, regularizers
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber

# Reproducibility
SEED = 42
N_SPLITS = 5
BATCH_SIZE = 32
EPOCHS = 800
PATIENCE = 60
LR = 5e-4
L2 = 1e-4
DROPOUT = 0.12
PI_COVERAGE = 0.90  # 90% nominal coverage for empirical prediction intervals

# Hyperparameters (Appendix A.2)
RF_PARAMS = dict(n_estimators=900, max_depth=20, min_samples_leaf=3,
                 max_features="sqrt", random_state=SEED, n_jobs=-1)
XGB_PARAMS = dict(n_estimators=900, max_depth=5, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
                  reg_alpha=0.1, random_state=SEED, tree_method='hist')
RIDGE_ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]
ENET_PARAMS = dict(alpha=1.0, l1_ratio=0.5, max_iter=10000, random_state=SEED)

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism(True)
except Exception:
    pass

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = f'FULL_RERUN_{timestamp}'
os.makedirs(OUT, exist_ok=True)
print("Setup complete. Output directory:", OUT)

In [ ]:
# Load training and test datasets
#
# Place training_dataset_lit.csv and testing_dataset_lit.csv in the same
# folder as this notebook, or adjust the paths below.

TRAIN_PATH = 'training_dataset_lit.csv'
TEST_PATH = 'testing_dataset_lit.csv'

# If the repository's data/ folder is one level up, this fallback finds it
for cand in [TRAIN_PATH, f'data/{TRAIN_PATH}', f'../data/{TRAIN_PATH}']:
    if os.path.exists(cand):
        TRAIN_PATH = cand; break
for cand in [TEST_PATH, f'data/{TEST_PATH}', f'../data/{TEST_PATH}']:
    if os.path.exists(cand):
        TEST_PATH = cand; break

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

# Sanity check on a known sample (OB-6 EC value, post-correction)
ob6 = train_df[train_df['Sample ID'] == 'OB-6']
if len(ob6) > 0:
    print(f"\nOB-6 sanity check: EC={ob6['EC (microS/cm)'].values[0]}, Li={ob6['Li (mg/l)'].values[0]}")
    assert ob6['EC (microS/cm)'].values[0] > 5000, "OB-6 EC unexpectedly low — check the dataset."
    print("OB-6 EC value verified.")

In [ ]:
# Column-name normalization and feature engineering

def normalize_colnames(df):
    """Normalize Unicode in column headers (µ → u, ° → deg, whitespace collapse)."""
    cols = []
    for c in df.columns:
        s = c.replace('\u00b5','u').replace('\u03bc','u').replace('\u00b0','deg')
        s = re.sub(r'\s+',' ', s).strip()
        cols.append(s)
    out = df.copy(); out.columns = cols
    return out

def pick(df, wanted, alts=None):
    """Robust column lookup with case- and punctuation-insensitive fallback."""
    alts = alts or []
    if wanted in df.columns: return wanted
    for a in alts:
        if a in df.columns: return a
    def canon(x): return re.sub(r'[^a-z0-9]', '', x.lower())
    w = canon(wanted)
    for c in df.columns:
        if canon(c) == w: return c
    raise KeyError(f"Column not found: {wanted}")

train_n = normalize_colnames(train_df)
test_n  = normalize_colnames(test_df)

# Eight base hydrogeochemical features (Section 3.2)
feat_names = [
    pick(train_n, 'Reservoir temperature (degC)', ['Reservoir temperature (°C)']),
    pick(train_n, 'pH'),
    pick(train_n, 'EC (uS/cm)', ['EC (microS/cm)']),
    pick(train_n, 'K (mg/l)'),
    pick(train_n, 'Na (mg/l)'),
    pick(train_n, 'Boron (mg/l)'),
    pick(train_n, 'SiO2 (mg/l)'),
    pick(train_n, 'Cl (mg/l)'),
]
target = pick(train_n, 'Li (mg/l)')
mg_col = pick(train_n, 'Mg (mg/l)')
group_col = 'Location'
groups = train_n[group_col].astype(str).values
y_train_raw = train_n[target].astype(float).values
y_test_raw  = test_n[target].astype(float).values

def build_features(df, include_mg=False, mg_mode='raw', feature_mode='combined'):
    """Construct feature matrix: 8 raw + 3 engineered ratios (Na/K, Na+K, Cl/EC).

    feature_mode: 'raw_only' (8) | 'engineered_only' (8) | 'combined' (11, default)
    """
    X = df[feat_names].astype(float).copy()
    Na = X[feat_names[4]].values; K = X[feat_names[3]].values
    EC = X[feat_names[2]].values; Cl = X[feat_names[7]].values
    if feature_mode in ('engineered_only', 'combined'):
        X['Na_K_ratio'] = np.clip((Na+1e-6)/(K+1e-6), 0.01, 100.0)
        X['Na_plus_K']  = Na + K
        X['Cl_over_EC'] = np.clip(Cl/(EC+1e-6), 1e-4, 1.0)
    if feature_mode == 'engineered_only':
        keep = [feat_names[0], feat_names[1], feat_names[2],
                'Na_K_ratio', 'Na_plus_K', 'Cl_over_EC', feat_names[5], feat_names[6]]
        X = X[[c for c in keep if c in X.columns]]
    if include_mg:
        mg = df[mg_col].astype(float).values
        if mg_mode == 'log': X['Mg_log'] = np.log1p(mg)
        elif mg_mode == 'binary': X['Mg_deep'] = (mg < 1.0).astype(float)
        else: X['Mg_raw'] = mg
    return X

X_train_raw = build_features(train_n)
X_test_raw  = build_features(test_n)
feature_names_ext = list(X_train_raw.columns)
print(f"Features ({len(feature_names_ext)}): {feature_names_ext}")

In [ ]:
# Metric and DNN utility functions

def scores(y_true, y_hat):
    """Return (RMSE, MAE, R²)."""
    return (np.sqrt(mean_squared_error(y_true, y_hat)),
            mean_absolute_error(y_true, y_hat),
            r2_score(y_true, y_hat))

def full_scores(y_true, y_hat):
    """Return (RMSE, MAE, R², MAPE%)."""
    rmse = np.sqrt(mean_squared_error(y_true, y_hat))
    mae = mean_absolute_error(y_true, y_hat)
    r2 = r2_score(y_true, y_hat)
    mape = np.mean(np.abs((y_true - y_hat) / np.maximum(y_true, 1e-6))) * 100
    return rmse, mae, r2, mape

def build_dnn(input_dim):
    """Compact feed-forward network: [input_dim] → 64 → 64 → 32 → 1 (Appendix A.2.1)."""
    m = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation='elu', kernel_regularizer=regularizers.l2(L2)), Dropout(DROPOUT),
        Dense(64, activation='elu', kernel_regularizer=regularizers.l2(L2)), Dropout(DROPOUT),
        Dense(32, activation='elu', kernel_regularizer=regularizers.l2(L2)),
        Dense(1)])
    m.compile(optimizer=Adam(learning_rate=LR, clipnorm=1.0), loss=Huber(delta=1.0), metrics=['mae'])
    return m

def inv_dnn(pred_std, mu, sigma):
    """Inverse-transform DNN output: standardized log1p → mg/L. Clips to [0, ∞)."""
    return np.maximum(np.expm1(np.maximum(pred_std * sigma + mu, 0.0)), 0.0)

def pi_stats(y_true, y_hat, q):
    """Empirical prediction interval coverage and mean width for half-width q."""
    lo, hi = y_hat - q, y_hat + q
    return lo, hi, np.mean((y_true >= lo) & (y_true <= hi)), np.mean(hi - lo)

print("Helpers ready.")

## Part A — Main GroupKFold cross-validation

Five-fold GroupKFold over the 26 location groups (Section 3.4). For each fold,
trains DNN, RF, XGB, Ridge, and ElasticNet baselines on raw or standardized
features as appropriate, and stores both validation and test predictions for
the per-sample-median fold-ensemble (Section 3.5).

In [ ]:
splitter = GroupKFold(n_splits=N_SPLITS)
folds_list = list(splitter.split(X_train_raw, y_train_raw, groups=groups))

cv_rows = []
early_stop_epochs = []
histories = []
all_models = ['DNN','RF','XGB','Ridge','ElasticNet']
perfold_q = {m: [] for m in all_models}
perfold_r2 = {m: [] for m in all_models}
test_preds = {m: [] for m in all_models}
val_preds_per_fold = {m: [] for m in all_models}

for fold_id, (tr_idx, val_idx) in enumerate(folds_list, 1):
    print(f"\n{'='*50}")
    print(f"FOLD {fold_id} (train={len(tr_idx)}, val={len(val_idx)}, val_groups={list(np.unique(groups[val_idx]))})")
    print(f"{'='*50}")

    X_tr = X_train_raw.iloc[tr_idx].values; X_va = X_train_raw.iloc[val_idx].values
    y_tr = y_train_raw[tr_idx]; y_va = y_train_raw[val_idx]
    X_te = X_test_raw.values

    # DNN: standardize inputs, log1p target with per-fold mean/std
    xsc = StandardScaler()
    X_tr_s = xsc.fit_transform(X_tr); X_va_s = xsc.transform(X_va); X_te_s = xsc.transform(X_te)
    y_tr_log = np.log1p(y_tr); mu = y_tr_log.mean(); sigma = y_tr_log.std() + 1e-8
    y_tr_std = (y_tr_log - mu) / sigma; y_va_std = (np.log1p(y_va) - mu) / sigma
    dnn = build_dnn(X_tr_s.shape[1])
    es = EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0)
    rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=30, min_lr=1e-6, verbose=0)
    hist = dnn.fit(X_tr_s, y_tr_std, validation_data=(X_va_s, y_va_std),
                   epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, callbacks=[es, rlr])
    histories.append(hist.history)
    early_stop_epochs.append(len(hist.history['loss']))
    y_va_dnn = inv_dnn(dnn.predict(X_va_s, verbose=0).ravel(), mu, sigma)
    y_te_dnn = inv_dnn(dnn.predict(X_te_s, verbose=0).ravel(), mu, sigma)
    print(f"  DNN: {len(hist.history['loss'])} epochs")

    # Random Forest on raw features
    rf = RandomForestRegressor(**RF_PARAMS); rf.fit(X_tr, np.log1p(y_tr))
    y_va_rf = np.expm1(rf.predict(X_va)); y_te_rf = np.expm1(rf.predict(X_te))

    # XGBoost on raw features
    xgb = XGBRegressor(**XGB_PARAMS); xgb.fit(X_tr, np.log1p(y_tr))
    y_va_xgb = np.expm1(xgb.predict(X_va)); y_te_xgb = np.expm1(xgb.predict(X_te))

    # Ridge on standardized features
    ridge = RidgeCV(alphas=RIDGE_ALPHAS, scoring='neg_mean_squared_error')
    ridge.fit(X_tr_s, np.log1p(y_tr))
    y_va_ridge = np.maximum(np.expm1(ridge.predict(X_va_s)), 0.0)
    y_te_ridge = np.maximum(np.expm1(ridge.predict(X_te_s)), 0.0)

    # ElasticNet on standardized features
    enet = ElasticNet(**ENET_PARAMS); enet.fit(X_tr_s, np.log1p(y_tr))
    y_va_enet = np.maximum(np.expm1(enet.predict(X_va_s)), 0.0)
    y_te_enet = np.maximum(np.expm1(enet.predict(X_te_s)), 0.0)

    # Store everything
    preds_map = {'DNN': (y_va_dnn, y_te_dnn), 'RF': (y_va_rf, y_te_rf),
                 'XGB': (y_va_xgb, y_te_xgb), 'Ridge': (y_va_ridge, y_te_ridge),
                 'ElasticNet': (y_va_enet, y_te_enet)}
    for m, (yv, yt) in preds_map.items():
        rmse, mae, r2 = scores(y_va, yv)
        cv_rows.append([m, fold_id, rmse, mae, r2, len(val_idx)])
        perfold_r2[m].append(r2)
        resid = np.abs(y_va - yv)
        perfold_q[m].append(np.quantile(resid, PI_COVERAGE))
        test_preds[m].append(yt)
        val_preds_per_fold[m].append((val_idx, yv, y_va))

print("\nCV loop complete.")

### Paper tables (3, 4, 5)

In [ ]:
cv_df = pd.DataFrame(cv_rows, columns=['model','fold','rmse','mae','r2','n_val'])
cv_df.to_csv(os.path.join(OUT, 'cv_metrics_all.csv'), index=False)

# Empirical prediction interval half-widths (median over fold-wise 90th-percentile residuals)
q_hat = {m: float(np.median(perfold_q[m])) for m in all_models}

# Ensemble test predictions: per-sample median over folds
test_results = {}
for m in all_models:
    y_pred = np.median(np.vstack(test_preds[m]), axis=0)
    rmse, mae, r2, mape = full_scores(y_test_raw, y_pred)
    _, _, cov, width = pi_stats(y_test_raw, y_pred, q_hat[m])
    test_results[m] = {'preds': y_pred, 'rmse': rmse, 'mae': mae, 'r2': r2,
                        'mape': mape, 'cov': cov, 'width': width}

# Table 3 — test-set metrics
print("="*70)
print("TABLE 3 — Test-set metrics")
print("="*70)
t3_rows = []
for m in ['DNN','Ridge','RF','XGB']:
    r = test_results[m]
    t3_rows.append([m, f"{r['rmse']:.3f}", f"{r['mae']:.3f}", f"{r['r2']:.3f}",
                    f"{r['mape']:.1f}", f"{r['cov']*100:.1f}%", f"{r['width']:.3f}"])
t3 = pd.DataFrame(t3_rows, columns=['Model','RMSE (mg/L)','MAE (mg/L)','R²','MAPE (%)','Empirical Coverage','Mean Width (mg/L)'])
print(t3.to_string(index=False))
t3.to_csv(os.path.join(OUT, 'TABLE3_test_metrics.csv'), index=False)

# Table 4 — per-fold CV
print("\n" + "="*70)
print("TABLE 4 — Per-fold CV metrics")
print("="*70)
for m in ['DNN','RF','XGB','Ridge']:
    sub = cv_df[cv_df['model']==m]
    print(f"\n{m}:")
    print(sub[['fold','rmse','mae','r2','n_val']].round(3).to_string(index=False))
cv_df.to_csv(os.path.join(OUT, 'TABLE4_cv_perfold.csv'), index=False)

# Table 5 — CV summary mean ± std
print("\n" + "="*70)
print("TABLE 5 — CV summary (mean ± std)")
print("="*70)
t5_rows = []
for m in ['DNN','RF','XGB','Ridge']:
    sub = cv_df[cv_df['model']==m]
    t5_rows.append([m,
        f"{sub['rmse'].mean():.3f} ± {sub['rmse'].std():.3f}",
        f"{sub['mae'].mean():.3f} ± {sub['mae'].std():.3f}",
        f"{sub['r2'].mean():.3f} ± {sub['r2'].std():.3f}"])
t5 = pd.DataFrame(t5_rows, columns=['Model','RMSE (mean±std)','MAE (mean±std)','R² (mean±std)'])
print(t5.to_string(index=False))
t5.to_csv(os.path.join(OUT, 'TABLE5_cv_summary.csv'), index=False)

## Part B — Manuscript figures

Generates the data-dependent figures: parity plots (Fig. 10), residual
diagnostics (Fig. 11), correlation heatmap (Fig. 8), SHAP beeswarm (Fig. 12),
and supplementary DNN training curves. Figure 9 (workflow schematic) is
produced separately by `scripts/make_fig9_workflow.py`.

In [ ]:
# Parity plots — all four models (Fig. 10)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, m in enumerate(['DNN', 'Ridge', 'RF', 'XGB']):
    y_pred = test_results[m]['preds']
    r = test_results[m]
    ax = axes[i]
    ax.scatter(y_test_raw, y_pred, alpha=0.85, s=70, zorder=3)
    lo = min(y_test_raw.min(), y_pred.min()) * 0.9
    hi = max(y_test_raw.max(), y_pred.max()) * 1.05
    ax.plot([lo,hi],[lo,hi],'r--', alpha=0.7, zorder=2)
    ax.set_xlabel('True Li (mg/L)', fontsize=12)
    if i == 0: ax.set_ylabel('Predicted Li (mg/L)', fontsize=12)
    ax.set_title(f'({chr(97+i)}) {m}\nRMSE={r["rmse"]:.2f}, R²={r["r2"]:.3f}', fontsize=11)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'Fig10_parity_4models.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Residual diagnostics — all four models (Fig. 11)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for i, m in enumerate(['DNN', 'Ridge', 'RF', 'XGB']):
    y_pred = test_results[m]['preds']
    resid = y_test_raw - y_pred
    ax_top = axes[0, i]
    ax_top.hist(resid, bins=10, alpha=0.8, edgecolor='black')
    ax_top.set_xlabel('Residual (mg/L)')
    if i == 0: ax_top.set_ylabel('Count')
    ax_top.set_title(f'{m}: Residual Histogram')
    ax_bot = axes[1, i]
    ax_bot.scatter(y_pred, resid, alpha=0.8, s=60)
    ax_bot.axhline(0, ls='--', c='r', alpha=0.7)
    ax_bot.set_xlabel('Predicted Li (mg/L)')
    if i == 0: ax_bot.set_ylabel('Residual (mg/L)')
    ax_bot.set_title(f'{m}: Residual vs Predicted')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'Fig11_residuals_4models.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (Fig. 8)
corr = np.corrcoef(X_train_raw.values.T)
plt.figure(figsize=(9,7))
im = plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(feature_names_ext)), feature_names_ext, rotation=90, fontsize=9)
plt.yticks(range(len(feature_names_ext)), feature_names_ext, fontsize=9)
plt.title('Feature Correlation Heatmap (Training Set)')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'Fig8_corr_heatmap.png'), dpi=300)
plt.show()

In [ ]:
# SHAP beeswarm — XGB (Fig. 12)
xgb_full = XGBRegressor(**XGB_PARAMS)
xgb_full.fit(X_train_raw.values, np.log1p(y_train_raw))
explainer = shap.TreeExplainer(xgb_full)
shap_vals = explainer.shap_values(X_train_raw.values)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, X_train_raw.values, feature_names=feature_names_ext, show=False)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'Fig12_shap_beeswarm.png'), dpi=300, bbox_inches='tight')
plt.close()
print('SHAP beeswarm saved.')

In [ ]:
# DNN training curves (supplementary)
fig, axes = plt.subplots(1, N_SPLITS, figsize=(4*N_SPLITS, 3.5))
for i, h in enumerate(histories):
    ax = axes[i]
    ax.plot(h['loss'], label='train', alpha=0.7)
    ax.plot(h['val_loss'], label='val', alpha=0.7)
    ax.set_title(f'Fold {i+1} ({len(h["loss"])} ep.)')
    ax.set_xlabel('Epoch'); ax.legend(fontsize=8)
    if i == 0: ax.set_ylabel('Loss')
plt.suptitle('DNN Training Curves (Early Stopping)', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'FigA_dnn_training_curves.png'), dpi=200, bbox_inches='tight')
plt.show()
print('All main figures saved.')

## Part C — Reviewer-requested experiments

### C1: GroupKFold diagnostics

In [ ]:
print("="*60)
print("GROUPKFOLD DIAGNOSTICS")
print("="*60)
unique_groups = np.unique(groups)
print(f"Unique groups: {len(unique_groups)}")
grp_counts = pd.Series(groups).value_counts().sort_values(ascending=False)
print(f"\nGroup sizes:\n{grp_counts.to_string()}")

print(f"\nFold assignments:")
for fold_i, (tr_i, va_i) in enumerate(folds_list, 1):
    vg = sorted(np.unique(groups[va_i]))
    print(f"  Fold {fold_i}: val n={len(va_i)}, groups={vg}")

print(f"\nTest set: n={len(y_test_raw)}")
print(f"  Locations: {list(test_n['Location'].values)}")
print(f"  Li range: {y_test_raw.min():.2f} - {y_test_raw.max():.2f} mg/L")

### C2: DNN complexity report

In [ ]:
n_feat = X_train_raw.shape[1]
p1 = (n_feat+1)*64; p2 = (64+1)*64; p3 = (64+1)*32; p4 = (32+1)*1
total_params = p1+p2+p3+p4
n_train_fold = int(len(y_train_raw) * (N_SPLITS-1) / N_SPLITS)

print("="*60)
print("DNN EFFECTIVE COMPLEXITY")
print("="*60)
print(f"Architecture: [{n_feat}] -> 64 -> 64 -> 32 -> 1")
print(f"Nominal parameters: {total_params:,}")
print(f"Regularization: L2={L2}, Dropout={DROPOUT}, EarlyStop patience={PATIENCE}")
print(f"Early stopping epochs: {early_stop_epochs}")
print(f"  Mean: {np.mean(early_stop_epochs):.0f}, Range: {min(early_stop_epochs)}-{max(early_stop_epochs)} / {EPOCHS} max")
print(f"Samples per fold: ~{n_train_fold}")
print(f"Samples/params ratio: {n_train_fold/total_params:.4f}")

### C3: DNN vs Ridge bootstrap test and LOO sensitivity

In [ ]:
y_pred_dnn = test_results['DNN']['preds']
y_pred_ridge = test_results['Ridge']['preds']

abs_err_dnn = np.abs(y_test_raw - y_pred_dnn)
abs_err_ridge = np.abs(y_test_raw - y_pred_ridge)
obs_diff = abs_err_ridge.mean() - abs_err_dnn.mean()

# Paired bootstrap
rng = np.random.RandomState(SEED)
boot_diffs = np.array([
    abs_err_ridge[rng.choice(len(y_test_raw), len(y_test_raw), replace=True)].mean() -
    abs_err_dnn[rng.choice(len(y_test_raw), len(y_test_raw), replace=True)].mean()
    for _ in range(10000)])
p_boot = np.mean(boot_diffs <= 0)
ci_lo, ci_hi = np.percentile(boot_diffs, [2.5, 97.5])

print("="*60)
print("PAIRED BOOTSTRAP: DNN vs Ridge")
print("="*60)
print(f"Observed MAE diff (Ridge-DNN): {obs_diff:.4f} mg/L")
print(f"Bootstrap 95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"Bootstrap p-value: {p_boot:.4f}")
sig = "SIGNIFICANT" if (p_boot < 0.05 and ci_lo > 0) else "NOT significant"
print(f"=> DNN advantage is {sig} at alpha=0.05")

# LOO sensitivity
print("\n" + "="*60)
print("LEAVE-ONE-OUT SENSITIVITY")
print("="*60)
loo_rows = []
sid_col = 'Sample ID' if 'Sample ID' in test_n.columns else None
for i in range(len(y_test_raw)):
    mask = np.ones(len(y_test_raw), dtype=bool); mask[i] = False
    r2d = r2_score(y_test_raw[mask], y_pred_dnn[mask])
    r2r = r2_score(y_test_raw[mask], y_pred_ridge[mask])
    sid = test_n.iloc[i][sid_col] if sid_col else f"s{i}"
    loo_rows.append([sid, y_test_raw[i], r2d, r2r, r2d-r2r])
loo_df = pd.DataFrame(loo_rows, columns=['Sample','Li_true','DNN_R2','Ridge_R2','Diff'])
print(loo_df.to_string(index=False, float_format='%.4f'))
loo_df.to_csv(os.path.join(OUT, 'loo_sensitivity.csv'), index=False)

tuzla_i = np.argmax(y_test_raw)
print(f"\nHighest-Li: {loo_df.iloc[tuzla_i]['Sample']} (Li={y_test_raw[tuzla_i]:.1f})")
print(f"  Without it: DNN R²={loo_df.iloc[tuzla_i]['DNN_R2']:.4f}, Ridge R²={loo_df.iloc[tuzla_i]['Ridge_R2']:.4f}")

### C4: Fold 1 diagnostics

In [ ]:
tr_idx, val_idx = folds_list[0]
y_va = y_train_raw[val_idx]

print("="*60)
print("FOLD 1 DIAGNOSTICS")
print("="*60)
print(f"Val groups: {sorted(np.unique(groups[val_idx]))}")
val_info = train_n.iloc[val_idx][['Sample ID','Location',target]].sort_values(target, ascending=False)
print(f"\nVal samples (sorted by Li):")
print(val_info.to_string(index=False))
print(f"\nTrain Li: max={y_train_raw[tr_idx].max():.1f}, mean={y_train_raw[tr_idx].mean():.2f}")
print(f"Val   Li: max={y_va.max():.1f}, mean={y_va.mean():.2f}")

# Per-model predictions on fold 1
X_tr_f = X_train_raw.iloc[tr_idx]; X_va_f = X_train_raw.iloc[val_idx]; y_tr_f = y_train_raw[tr_idx]
xsc = StandardScaler(); X_tr_s = xsc.fit_transform(X_tr_f.values); X_va_s = xsc.transform(X_va_f.values)
y_log = np.log1p(y_tr_f); mu = y_log.mean(); sig = y_log.std()+1e-8

dnn_f = build_dnn(X_tr_s.shape[1])
dnn_f.fit(X_tr_s, (y_log-mu)/sig, validation_data=(X_va_s, (np.log1p(y_va)-mu)/sig),
          epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
          callbacks=[EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0)])
p_dnn = inv_dnn(dnn_f.predict(X_va_s, verbose=0).ravel(), mu, sig)

rf_f = RandomForestRegressor(**RF_PARAMS); rf_f.fit(X_tr_f.values, np.log1p(y_tr_f))
p_rf = np.expm1(rf_f.predict(X_va_f.values))

xgb_f = XGBRegressor(**XGB_PARAMS); xgb_f.fit(X_tr_f.values, np.log1p(y_tr_f))
p_xgb = np.expm1(xgb_f.predict(X_va_f.values))

diag = pd.DataFrame({'Sample': train_n.iloc[val_idx]['Sample ID'].values,
    'Location': train_n.iloc[val_idx]['Location'].values,
    'Li_true': y_va, 'DNN': p_dnn, 'RF': p_rf, 'XGB': p_xgb})
diag = diag.sort_values('Li_true', ascending=False)
for m in ['DNN','RF','XGB']: diag[f'{m}_err'] = diag['Li_true'] - diag[m]
print("\nPer-sample predictions:")
print(diag.to_string(index=False, float_format='%.2f'))
diag.to_csv(os.path.join(OUT, 'fold1_diagnostics.csv'), index=False)

# R² with/without max-Li sample
mx = np.argmax(y_va); msk = np.ones(len(y_va), dtype=bool); msk[mx] = False
print(f"\nHighest-Li in Fold 1: {y_va[mx]:.1f} mg/L ({train_n.iloc[val_idx[mx]]['Sample ID']})")
print(f"R² WITH:    DNN={r2_score(y_va, p_dnn):.3f}, RF={r2_score(y_va, p_rf):.3f}, XGB={r2_score(y_va, p_xgb):.3f}")
if msk.sum() > 1:
    print(f"R² WITHOUT: DNN={r2_score(y_va[msk], p_dnn[msk]):.3f}, RF={r2_score(y_va[msk], p_rf[msk]):.3f}, XGB={r2_score(y_va[msk], p_xgb[msk]):.3f}")

### C5: Mg²⁺ ablation

In [ ]:
print("="*60)
print("Mg²⁺ ABLATION")
print("="*60)
mg_rows = []
for label, inc, mode in [('No Mg',False,None), ('Mg raw',True,'raw'), ('Mg log',True,'log'), ('Mg binary',True,'binary')]:
    Xtr = build_features(train_n, include_mg=inc, mg_mode=mode)
    Xte = build_features(test_n, include_mg=inc, mg_mode=mode)
    for mname, Model, params in [('RF', RandomForestRegressor, RF_PARAMS), ('XGB', XGBRegressor, XGB_PARAMS)]:
        fold_preds = []; fold_r2s = []
        for tr_i, va_i in folds_list:
            m = Model(**params); m.fit(Xtr.iloc[tr_i].values, np.log1p(y_train_raw[tr_i]))
            fold_preds.append(np.expm1(m.predict(Xte.values)))
            fold_r2s.append(r2_score(y_train_raw[va_i], np.expm1(m.predict(Xtr.iloc[va_i].values))))
        y_ens = np.median(np.vstack(fold_preds), axis=0)
        rmse, mae, r2, _ = full_scores(y_test_raw, y_ens)
        mg_rows.append([label, mname, Xtr.shape[1], rmse, mae, r2, np.mean(fold_r2s)])
mg_df = pd.DataFrame(mg_rows, columns=['Mg_config','Model','n_feat','RMSE','MAE','R2','CV_R2_mean'])
print(mg_df.round(4).to_string(index=False))
mg_df.to_csv(os.path.join(OUT, 'mg_ablation.csv'), index=False)

### C6: Feature engineering ablation

In [ ]:
print("="*60)
print("FEATURE ENGINEERING ABLATION")
print("="*60)
fa_rows = []
for label, fmode in [('raw_only (8)', 'raw_only'), ('engineered_only (8)', 'engineered_only'), ('combined (11)', 'combined')]:
    Xtr = build_features(train_n, feature_mode=fmode)
    Xte = build_features(test_n, feature_mode=fmode)
    print(f"\n{label}: {list(Xtr.columns)}")
    for mname, Model, params in [('RF', RandomForestRegressor, RF_PARAMS), ('XGB', XGBRegressor, XGB_PARAMS),
                                  ('DNN', None, None)]:
        fold_preds = []; fold_r2s = []
        for tr_i, va_i in folds_list:
            if mname == 'DNN':
                sc = StandardScaler()
                Xtr_s = sc.fit_transform(Xtr.iloc[tr_i].values); Xva_s = sc.transform(Xtr.iloc[va_i].values)
                Xte_s = sc.transform(Xte.values)
                yl = np.log1p(y_train_raw[tr_i]); mu_ = yl.mean(); si_ = yl.std()+1e-8
                d = build_dnn(Xtr_s.shape[1])
                d.fit(Xtr_s, (yl-mu_)/si_, validation_data=(Xva_s, (np.log1p(y_train_raw[va_i])-mu_)/si_),
                      epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
                      callbacks=[EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0)])
                fold_preds.append(inv_dnn(d.predict(Xte_s, verbose=0).ravel(), mu_, si_))
                fold_r2s.append(r2_score(y_train_raw[va_i], inv_dnn(d.predict(Xva_s, verbose=0).ravel(), mu_, si_)))
            else:
                m = Model(**params); m.fit(Xtr.iloc[tr_i].values, np.log1p(y_train_raw[tr_i]))
                fold_preds.append(np.expm1(m.predict(Xte.values)))
                fold_r2s.append(r2_score(y_train_raw[va_i], np.expm1(m.predict(Xtr.iloc[va_i].values))))
        y_ens = np.median(np.vstack(fold_preds), axis=0)
        rmse, mae, r2, _ = full_scores(y_test_raw, y_ens)
        fa_rows.append([label, mname, Xtr.shape[1], rmse, mae, r2, np.mean(fold_r2s)])
fa_df = pd.DataFrame(fa_rows, columns=['Config','Model','n_feat','RMSE','MAE','R2','CV_R2_mean'])
print("\n" + fa_df.round(4).to_string(index=False))
fa_df.to_csv(os.path.join(OUT, 'feature_ablation.csv'), index=False)

### C7: SiO₂/T ablation

In [ ]:
print("="*60)
print("SiO₂/T ABLATION")
print("="*60)
sio2_rows = []
for label, add_sio2t in [('without SiO2/T', False), ('with SiO2/T', True)]:
    Xtr = build_features(train_n); Xte = build_features(test_n)
    if add_sio2t:
        T_tr = train_n[feat_names[0]].astype(float).values; S_tr = train_n[feat_names[6]].astype(float).values
        Xtr['SiO2_T'] = S_tr / (T_tr + 1e-6)
        T_te = test_n[feat_names[0]].astype(float).values; S_te = test_n[feat_names[6]].astype(float).values
        Xte['SiO2_T'] = S_te / (T_te + 1e-6)
    for mname, Model, params in [('RF', RandomForestRegressor, RF_PARAMS), ('XGB', XGBRegressor, XGB_PARAMS)]:
        fold_preds = []; fold_r2s = []
        for tr_i, va_i in folds_list:
            m = Model(**params); m.fit(Xtr.iloc[tr_i].values, np.log1p(y_train_raw[tr_i]))
            fold_preds.append(np.expm1(m.predict(Xte.values)))
            fold_r2s.append(r2_score(y_train_raw[va_i], np.expm1(m.predict(Xtr.iloc[va_i].values))))
        y_ens = np.median(np.vstack(fold_preds), axis=0)
        rmse, mae, r2, _ = full_scores(y_test_raw, y_ens)
        sio2_rows.append([label, mname, rmse, r2, np.mean(fold_r2s)])
s_df = pd.DataFrame(sio2_rows, columns=['Config','Model','RMSE','R2','CV_R2_mean'])
print(s_df.round(4).to_string(index=False))
s_df.to_csv(os.path.join(OUT, 'sio2t_ablation.csv'), index=False)

### C8: Empirical coverage binomial confidence interval

In [ ]:
print("="*60)
print("EMPIRICAL COVERAGE: BINOMIAL CI (n=12)")
print("="*60)
n = 12
for k in [10, 11, 12]:
    ci_lo_ = stats.beta.ppf(0.025, k, n-k+1) if k > 0 else 0.0
    ci_hi_ = stats.beta.ppf(0.975, k+1, n-k) if k < n else 1.0
    print(f"  k={k}/{n} ({k/n*100:.1f}%): 95% Clopper-Pearson CI = [{ci_lo_*100:.1f}%, {ci_hi_*100:.1f}%]")
k_obs = 11
ci_l = stats.beta.ppf(0.025, k_obs, n-k_obs+1)
ci_h = stats.beta.ppf(0.975, k_obs+1, n-k_obs)
print(f"\nFor paper: 91.7% observed coverage, 95% CI [{ci_l*100:.0f}%, {ci_h*100:.0f}%]")

### C9: Mg²⁺ completeness report

In [ ]:
mg_tr = train_n[mg_col].astype(float)
mg_te = test_n[mg_col].astype(float)
print(f"Train: {mg_tr.notna().sum()}/{len(mg_tr)} have Mg ({mg_tr.isna().sum()} missing)")
print(f"Test:  {mg_te.notna().sum()}/{len(mg_te)} have Mg ({mg_te.isna().sum()} missing)")
print(f"Train Mg < 1.0: {(mg_tr < 1.0).sum()}/{len(mg_tr)} ({(mg_tr < 1.0).mean()*100:.0f}%)")
print(f"Train Mg stats: min={mg_tr.min():.2f}, max={mg_tr.max():.2f}, median={mg_tr.median():.2f}")

## Part D — Headline numbers for the paper

Convenience summary for verifying the values quoted in the Abstract,
Section 4.2, and Conclusion.

In [ ]:
print("="*70)
print("HEADLINE NUMBERS")
print("="*70)

d = test_results['DNN']; ri = test_results['Ridge']
rf = test_results['RF']; xg = test_results['XGB']

print(f"""
SECTION 4.2 (Test-set performance):
  DNN:   RMSE={d['rmse']:.3f}, MAE={d['mae']:.3f}, R²={d['r2']:.3f}, MAPE={d['mape']:.1f}%
  Ridge: RMSE={ri['rmse']:.3f}, MAE={ri['mae']:.3f}, R²={ri['r2']:.3f}, MAPE={ri['mape']:.1f}%
  RF:    RMSE={rf['rmse']:.3f}, MAE={rf['mae']:.3f}, R²={rf['r2']:.3f}, MAPE={rf['mape']:.1f}%
  XGB:   RMSE={xg['rmse']:.3f}, MAE={xg['mae']:.3f}, R²={xg['r2']:.3f}, MAPE={xg['mape']:.1f}%

EMPIRICAL PREDICTION INTERVALS (90% nominal):
  DNN:   coverage={d['cov']*100:.1f}%, width={d['width']:.3f} mg/L
  Ridge: coverage={ri['cov']*100:.1f}%, width={ri['width']:.3f} mg/L
  RF:    coverage={rf['cov']*100:.1f}%, width={rf['width']:.3f} mg/L
  XGB:   coverage={xg['cov']*100:.1f}%, width={xg['width']:.3f} mg/L

BOOTSTRAP (DNN vs Ridge):
  MAE diff: {obs_diff:.4f}, 95% CI: [{ci_lo:.4f}, {ci_hi:.4f}], p={p_boot:.4f}
  {"SIGNIFICANT" if p_boot < 0.05 else "NOT significant"}

DNN COMPLEXITY:
  Parameters: {total_params:,}, Early stop mean {np.mean(early_stop_epochs):.0f} epochs
  ({min(early_stop_epochs)}-{max(early_stop_epochs)} range, {EPOCHS} max)
""")

## Part E — Save all predictions and outputs

Writes per-sample predictions with empirical prediction interval bounds
(`predictions_all.csv`). All numeric tables and figures from earlier cells
are already saved under the timestamped output directory.

In [ ]:
pred_data = {'y_true': y_test_raw}
for m in ['DNN','Ridge','RF','XGB']:
    pred_data[f'pred_{m}'] = test_results[m]['preds']
    pred_data[f'pi_lo_{m}'] = test_results[m]['preds'] - q_hat[m]
    pred_data[f'pi_hi_{m}'] = test_results[m]['preds'] + q_hat[m]
pd.DataFrame(pred_data).to_csv(os.path.join(OUT, 'predictions_all.csv'), index=False)

print(f"\nAll outputs saved to: {OUT}/")
for f in sorted(os.listdir(OUT)):
    print(f"  {f}")